In [ ]:
from pathlib import Path
import subprocess,sys,json,hashlib
import pandas as pd
import numpy as np
work=Path('/kaggle/working')
src=work/'cnx_solo_source';src.mkdir(exist_ok=True)
for name,text in {'preprocess.py': '"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].\n\nCanonical means: fixed physical pixel size, centre crop to a fixed field of view,\nstandard in-plane orientation, standard slice order and a per-series intensity window.\nThe same `load_series` is used for the training cache and for inference.\n\nUsage: python src/preprocess.py [train|test] [n_series_limit]\n"""\nimport os\nimport sys\nfrom multiprocessing import Pool\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\n\nDATA = "/mnt/ssd/rsna_knee/data"\nWORK = "/mnt/ssd/rsna_knee/work"\n\nMM_PER_PX = 0.4\nSIZE = 384  # 153.6 mm field of view\nMAX_SLICES = 64\n\n# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates\n# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior\n# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior\n# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior\nCANON = {\n    0: (+1, (1, +1), (2, -1)),\n    1: (+1, (0, +1), (2, -1)),\n    2: (-1, (0, +1), (1, +1)),\n}\nPLANES = ["Sagittal", "Coronal", "Axial"]\n\n\ndef _read(path):\n    ds = pydicom.dcmread(path)\n    arr = ds.pixel_array.astype(np.float32)\n    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":\n        arr = arr.max() - arr\n    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")\n    if slope is not None and inter is not None:\n        arr = arr * float(slope) + float(inter)\n    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None\n    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None\n    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None\n    inst = float(ds.get("InstanceNumber") or 0)\n    frames = list(arr) if arr.ndim == 3 else [arr]\n    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]\n\n\ndef _canon_slice(img, iop, ps, plane):\n    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""\n    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop\n    if iop is None:\n        return img, ps\n    r, c = np.array(iop[:3]), np.array(iop[3:])\n    _, (cax, csgn), (rax, rsgn) = CANON[plane]\n    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical\n        img, r, c, ps = img.T, c, r, ps[::-1]\n    if r[cax] * csgn < 0:\n        img = img[:, ::-1]\n    if c[rax] * rsgn < 0:\n        img = img[::-1]\n    return img, ps\n\n\ndef _resample(img, ps):\n    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""\n    out = np.zeros((SIZE, SIZE), np.float32)\n    h, w = img.shape\n    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))\n    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))\n    y0, x0 = (h - ch) // 2, (w - cw) // 2\n    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])\n    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))\n    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))\n    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR\n    res = cv2.resize(crop, (nw, nh), interpolation=interp)\n    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2\n    out[oy:oy + nh, ox:ox + nw] = res\n    return out\n\n\ndef load_series(sdir, plane_name=None):\n    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""\n    slices, bad = [], 0\n    for fn in os.listdir(sdir):\n        try:\n            slices.extend(_read(os.path.join(sdir, fn)))\n        except Exception:\n            bad += 1\n    info = {"n_raw": len(slices), "n_bad": bad}\n    if not slices:\n        return np.zeros((1, SIZE, SIZE), np.uint8), info\n\n    iop = next((s[1] for s in slices if s[1] is not None), None)\n    if iop is not None:\n        normal = np.cross(iop[:3], iop[3:])\n        plane = int(np.abs(normal).argmax())\n    else:\n        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0\n    info["plane"] = PLANES[plane]\n\n    if normal is not None and all(s[2] is not None for s in slices):\n        sgn = CANON[plane][0] * np.sign(normal[plane])\n        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)\n        pos = [float(np.dot(s[2], normal)) for s in slices]\n        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0\n    else:\n        slices.sort(key=lambda s: s[4])\n    if len(slices) > MAX_SLICES:\n        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)\n        slices = [slices[i] for i in keep]\n\n    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])\n    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])\n    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)\n    info["n"] = len(vol)\n    return (vol * 255).round().astype(np.uint8), info\n\n\ndef _job(args):\n    split, study, series, plane, out_dir = args\n    out = f"{out_dir}/{series}.npy"\n    info = {"study": study, "series": series}\n    if os.path.exists(out):\n        return None\n    try:\n        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)\n        np.save(out + ".tmp.npy", vol)\n        os.replace(out + ".tmp.npy", out)\n        info.update(extra)\n    except Exception as e:\n        info["error"] = f"{type(e).__name__}: {e}"[:200]\n    return info\n\n\ndef main():\n    split = sys.argv[1] if len(sys.argv) > 1 else "train"\n    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None\n    out_dir = f"{WORK}/cache{SIZE}_{split}"\n    os.makedirs(out_dir, exist_ok=True)\n    se = pd.read_csv(f"{DATA}/{split}_series.csv")\n    if limit:\n        se = se.sample(limit, random_state=0)\n    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]\n    rows = []\n    with Pool(18) as p:\n        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):\n            if r:\n                rows.append(r)\n            if i % 1000 == 0:\n                print(f"{i}/{len(tasks)}", flush=True)\n    df = pd.DataFrame(rows)\n    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"\n    if os.path.exists(meta) and len(df):\n        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")\n    if len(df):\n        df.to_csv(meta, index=False)\n        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)\n\n\nif __name__ == "__main__":\n    main()', 'knee.py': '"""Shared dataset and model code for training and inference."""\nimport math\n\nimport numpy as np\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n          "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nPLANES = ["Sagittal", "Coronal", "Axial"]\n# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first\nN_SLOTS = 6\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef slot_of(plane, fat_sup):\n    return PLANES.index(plane) + 3 * (1 - int(fat_sup))\n\n\ndef build_study_table(series_df, n_slices):\n    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.\n    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}\n    with each slot\'s candidates sorted by slice count (largest first)."""\n    table = {}\n    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,\n                              series_df.Fat_Suppression, series_df.Anatomical_Plane):\n        if n_slices.get(sr, 0) < 3:\n            continue\n        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)\n    for slots in table.values():\n        for cands in slots:\n            cands.sort(key=lambda s: -n_slices[s])\n    return table\n\n\ndef window_centres(n, k, train, lo=0.04, hi=0.96):\n    """k slice indices (centres of 3-slice windows) spread over the series."""\n    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))\n    if b < a:\n        a = b = (n - 1) / 2\n    if train:\n        edges = np.linspace(a, b, k + 1)\n        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])\n    else:\n        c = np.linspace(a, b, k)\n    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))\n\n\nclass StudyDataset(torch.utils.data.Dataset):\n    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""\n\n    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):\n        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir\n        self.targets, self.k, self.train = targets, k, train\n        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        st = self.studies[i]\n        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])\n        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        for s, cands in enumerate(slots):\n            if not cands:\n                continue\n            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]\n            vol = self.loader(sr)\n            n = len(vol)\n            c = window_centres(n, self.k, self.train)\n            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]\n            if x is None:\n                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)\n            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)\n        if x is None:\n            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)\n        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))\n\n\ndef make_views(x, res, train, group, fov=0.92):\n    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached\n    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter\n    that is shared within a series."""\n    n = x.shape[0]\n    ns = n // group\n    dev = x.device\n    x = x.float().div_(255.0)\n    if train:\n        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)\n        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)\n        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)\n        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2\n    else:\n        ang = torch.zeros(ns, device=dev)\n        sc = torch.full((ns,), fov, device=dev)\n        tx = ty = torch.zeros(ns, device=dev)\n        gamma = gain = None\n    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc\n    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)\n    theta = theta.repeat_interleave(group, 0)\n    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)\n    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot\n        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)\n    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    if train:\n        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)\n        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)\n    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\nclass KneeNet(nn.Module):\n    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->\n    small transformer over all windows of the study -> one attention pooling per finding."""\n\n    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,\n                 img_size=None):\n        super().__init__()\n        kw = {"drop_path_rate": drop_path} if drop_path else {}\n        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size\n            kw["img_size"] = img_size\n        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(N_SLOTS, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))\n        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))\n\n    def head(self, feat, mask, pos):\n        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""\n        b, s, k, _ = feat.shape\n        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)\n        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))\n        h = h.view(b, s * k, -1)\n        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)\n        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything\n        if self.tf is not None:\n            h = self.tf(h, src_key_padding_mask=pad)\n        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]\n        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n        return (pooled * self.cls_w).sum(-1) + self.cls_b\n\n    def forward(self, x, mask, pos, res=256, train=False):\n        """x uint8 [B, S, K, 3, H, W]."""\n        b, s, k = x.shape[:3]\n        keep = mask.view(-1)  # run the backbone only on series that exist\n        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)\n        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))\n        feat = f.new_zeros(b * s, k, f.shape[-1])\n        feat[keep] = f.view(-1, k, f.shape[-1])\n        return self.head(feat.view(b, s, k, -1), mask, pos)', 'infer.py': '"""Predict the 12 findings for every study straight from DICOM folders.\n\nUsed by the Kaggle submission notebook and locally to check that predictions from raw DICOMs\nmatch those from the training cache.\n\nUsage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]\n  data_dir contains <split>.csv, <split>_series.csv and <split>_series/\n"""\nimport os\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres\nfrom preprocess import MAX_SLICES, SIZE, load_series\n\n\nclass DicomStudyDataset(torch.utils.data.Dataset):\n    """Loads every selected series of a study once; windows are cut per model setting later."""\n\n    def __init__(self, studies, table, root, k):\n        self.studies, self.table, self.root, self.k = list(studies), table, root, k\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        st = self.studies[i]\n        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])\n        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)\n        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        for s, cands in enumerate(slots):\n            if not cands:\n                continue\n            vol, info = load_series(f"{self.root}/{st}/{cands[0]}")\n            if info.get("n_bad", 0):\n                raise RuntimeError(f"DICOM decoding failed: {st}/{cands[0]}: {info}")\n            n = len(vol)\n            if n < 3:\n                raise RuntimeError(f"Selected series has fewer than 3 valid slices: {st}/{cands[0]}")\n            c = window_centres(n, self.k, False)\n            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])\n            mask[s], pos[s] = True, c / max(n - 1, 1)\n        if not mask.any():\n            raise RuntimeError(f"Study has no usable series: {st}")\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)\n\n\ndef load_models(ckpts, device="cuda"):\n    models = []\n    for path in ckpts:\n        ck = torch.load(path, map_location="cpu", weights_only=True)\n        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions\n        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False, img_size=ck["args"]["res"])\n        m.load_state_dict(ck["model"], strict=True)\n        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))\n    return models\n\n\n@torch.no_grad()\ndef run(data_dir, split, ckpts, k=12, studies=None, workers=4, bs=4):\n    se = pd.read_csv(f"{data_dir}/{split}_series.csv")\n    root = f"{data_dir}/{split}_series"\n    if studies is None:\n        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()\n    se = se[se.StudyInstanceUID.isin(set(studies))]\n    n_files = {}\n    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):\n        d = f"{root}/{st}/{sr}"\n        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0\n    table = build_study_table(se, n_files)\n    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,\n                                     num_workers=workers, pin_memory=True)\n    models = load_models(ckpts)\n    out = []\n    for x, mask, pos in dl:\n        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()\n        with torch.autocast("cuda", dtype=torch.float16):\n            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)\n        out.append(p.cpu())\n    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)\n\n\nif __name__ == "__main__":\n    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]\n    run(data_dir, split, ckpts).to_csv(out_csv)\n    print("wrote", out_csv)', 'worker.py': '"""Isolated public reader execution; imported modules belong to this process only."""\nimport glob\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nstarted=time.monotonic()\nsrc=Path(__file__).resolve().parent\nroots=[p.parent for p in Path(\'/kaggle/input\').glob(\'datasets/*/*/cnxt_v0_fold0.pt\')]\nroots += [p.parent for p in Path(\'/kaggle/input\').glob(\'*/cnxt_v0_fold0.pt\')]\nassert len(set(roots))==1, \'Checkpoint root must be unique\'\nasset=roots[0]\nckpts=[asset/f\'cnxt_v0_fold{i}.pt\' for i in range(3)]\nassert all(p.is_file() and p.stat().st_size==123035271 for p in ckpts), \'All three exact weights required\'\nenv=Path(\'/kaggle/working/cnx3_env\')\nsubprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-index\',\'--no-deps\',\'--target\',str(env),\'--find-links\',str(asset),\'pylibjpeg\',\'pylibjpeg-libjpeg\',\'pylibjpeg-openjpeg\'],check=True)\ntimm_wheels=sorted(Path(\'/kaggle/input\').glob(\'datasets/*/*/timm-*.whl\'))+sorted(Path(\'/kaggle/input\').glob(\'*/timm-*.whl\'))\nassert timm_wheels, \'Offline timm wheel required\'\nsubprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-index\',\'--no-deps\',\'--target\',str(env),str(timm_wheels[0])],check=True)\nsys.path.insert(0,str(env))\nsys.path.insert(0,str(src))\nimport torch\nimport numpy as np\nimport pandas as pd\nimport infer\nfrom knee import LABELS, build_study_table\nfrom preprocess import MAX_SLICES\ntorch.set_num_threads(4)\ntorch.cuda.set_device(0)\nallocator_warmup=torch.empty(1,device=\'cuda:0\')\ndel allocator_warmup\ntorch.cuda.reset_peak_memory_stats(0)\ncomp=Path(\'/kaggle/input/competitions/rsna-knee-abnormality-detection\')\nif not comp.is_dir(): comp=Path(\'/kaggle/input/rsna-knee-abnormality-detection\')\ntest=pd.read_csv(comp/\'test.csv\',dtype={\'StudyInstanceUID\':str})\nids=test.StudyInstanceUID.tolist()\nassert ids and len(ids)==len(set(ids))\nse=pd.read_csv(comp/\'test_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\nse=se[se.StudyInstanceUID.isin(ids)]\nn_files={sr:min(len(os.listdir(comp/\'test_series\'/st/sr)),MAX_SLICES) for st,sr in zip(se.StudyInstanceUID,se.SeriesInstanceUID)}\ntable=build_study_table(se,n_files)\nds=infer.DicomStudyDataset(ids,table,str(comp/\'test_series\'),12)\nmodels=infer.load_models([str(p) for p in ckpts])\nassert len(models)==3\nconfig=[]\nfor p in ckpts:\n    ck=torch.load(p,map_location=\'cpu\',weights_only=True)\n    assert ck[\'args\'][\'backbone\'].split(\'.\')[0]==\'convnext_tiny\' and int(ck[\'args\'][\'res\'])==256, \'Unexpected architecture/input\'\n    config.append(ck[\'args\'])\n    del ck\n# Full batch and all six slots: memory/latency check only, discarded after use.\nx,mask,pos=ds[0]\nbx=x.unsqueeze(0).repeat(4,1,1,1,1,1).cuda()\nbm=torch.ones((4,6),dtype=torch.bool,device=\'cuda\')\nbp=pos.unsqueeze(0).repeat(4,1,1).cuda()\nstress_start=time.monotonic()\nwith torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.float16):\n    for model,res in models:\n        probe=model(bx,bm,bp,res=res)\n        assert probe.shape==(4,12) and torch.isfinite(probe).all()\ntorch.cuda.synchronize()\nstress_seconds=time.monotonic()-stress_start\nstress_peak=torch.cuda.max_memory_allocated(0)\ndel bx,bm,bp,probe\ntorch.cuda.empty_cache()\n# Keep loading in-process: no CUDA-after-fork or shared-memory prefetch queue.\n# Batch size, windows, pixels and predictions are unchanged.\ndl=torch.utils.data.DataLoader(ds,batch_size=4,shuffle=False,num_workers=0,pin_memory=True)\nprint(\'CNX3 inference start\',len(ids),\'studies; loader_workers=0\',flush=True)\nout=[];valid=[];inference_start=time.monotonic()\nwith torch.inference_mode():\n    for batch_index,(x,mask,pos) in enumerate(dl):\n        assert mask.any(1).all(), \'Empty study\'\n        valid.extend(mask.sum(1).tolist())\n        x,mask,pos=x.cuda(non_blocking=True),mask.cuda(),pos.cuda()\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            prob=torch.stack([m(x,mask,pos,res=res).float().sigmoid() for m,res in models]).mean(0)\n        assert torch.isfinite(prob).all()\n        out.append(prob.cpu().numpy())\n        if batch_index%25==0:\n            print(\'CNX3 progress\',len(valid),\'/\',len(ids),\'seconds\',time.monotonic()-inference_start,flush=True)\nvalues=np.concatenate(out)\nassert values.shape==(len(ids),12) and np.isfinite(values).all() and (values>=0).all() and (values<=1).all()\nframe=pd.DataFrame(values,columns=LABELS)\nframe.insert(0,\'StudyInstanceUID\',ids)\nwork=Path(\'/kaggle/working\')\npath=work/\'cnx3_raw.csv\'\nframe.to_csv(path,index=False)\nreceipt=dict(status=\'COMPLETE\',study_count=len(ids),folds=[0,1,2],model_count=3,\n    precision=\'fp16\',batch_studies=4,windows_per_slot=12,slots_present=valid,loader_workers=0,\n    checkpoint_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in ckpts},\n    checkpoint_args=config,stress_pass=True,stress_seconds=stress_seconds,stress_peak_gpu_bytes=stress_peak,\n    peak_gpu_bytes=torch.cuda.max_memory_allocated(0),elapsed_seconds=time.monotonic()-started,\n    inference_seconds=time.monotonic()-inference_start,source_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in src.glob(\'*.py\')},\n    raw_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),fallback=0,partial_predictions=0)\n(work/\'cnx3_worker_receipt.json\').write_text(json.dumps(receipt,indent=2,default=str))\nprint(\'CNX3 COMPLETE\',receipt,flush=True)\n'}.items(): (src/name).write_text(text,encoding='utf-8')
with (work/'solo_worker.log').open('w') as log:
    try:
        subprocess.run([sys.executable,'-u',str(src/'worker.py')],stdout=log,stderr=subprocess.STDOUT,check=True,timeout=2*3600)
    except Exception:
        print((work/'solo_worker.log').read_text()[-8000:])
        raise
receipt=json.loads((work/'cnx3_worker_receipt.json').read_text())
assert receipt['status']=='COMPLETE' and receipt['model_count']==3 and receipt['folds']==[0,1,2]
assert receipt['stress_pass'] and receipt['fallback']==0 and receipt['partial_predictions']==0
frame=pd.read_csv(work/'cnx3_raw.csv',dtype={'StudyInstanceUID':str})
comp=Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
if not comp.exists():comp=Path('/kaggle/input/rsna-knee-abnormality-detection')
ids=pd.read_csv(comp/'test.csv',dtype={'StudyInstanceUID':str}).StudyInstanceUID.tolist()
assert frame.StudyInstanceUID.tolist()==ids and frame.StudyInstanceUID.is_unique
assert frame.columns.tolist()==['StudyInstanceUID','ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
v=frame.iloc[:,1:].to_numpy();assert np.isfinite(v).all() and (v>=0).all() and (v<=1).all()
raw=(work/'cnx3_raw.csv').read_bytes()
assert hashlib.sha256(raw).hexdigest()==receipt['raw_sha256']
target=work/'submission.csv';tmp=work/'solo_submission.tmp';tmp.write_bytes(raw);tmp.replace(target)
receipt.update(candidate='threefold_reader_only',parent_used=False,submission_sha256=hashlib.sha256(raw).hexdigest())
(work/'solo_receipt.json').write_text(json.dumps(receipt,indent=2))
print('SOLO COMPLETE',len(frame),'studies',receipt['elapsed_seconds'],'seconds')
